In [1]:
#Librerías
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import mysql.connector
from math import floor
from scipy.stats import norm
from collections import defaultdict
from sklearn.impute import KNNImputer
import re

from io import BytesIO
import xlsxwriter
from tqdm import tqdm

import unicodedata

## Leer excel maestros

In [2]:
scala_maestro=pd.read_excel('Tabla Maestra SCALA 2.xlsx')
condado_maestro=pd.read_excel('Tabla maestro Condado 3.xlsx')

malls=pd.read_excel('avance_mall.xlsx')


In [3]:
scala_maestro.head(4)

,num,nombre_establecimiento,ruc,cod_establecimiento,Estado,CATEGORIA,cod_esablecimiento,id_establecimiento,data_brick,coinc_data_brick,indicad_data_brick,data_fact,coinc_data_fact,indicad_data_fact
0,1,ACIUM,1793193935001,2,ABIERTO,ACCESORIOS,002,1.793194e+15,1.023116e+14,NaN,0.0,1.023116e+14,58.0,1.0
1,2,ADIDAS,1792056055001,13,ABIERTO,MODA,013,1.792056e+15,NaN,NaN,0.0,NaN,NaN,0.0
2,3,ADOLFO DOMINGUEZ,1791836979001,37,ABIERTO,MODA,037,1.791837e+15,9.921069e+14,NaN,0.0,9.921069e+14,NaN,0.0
3,4,ALAJA BY CIRE,1792373379001,4,ABIERTO,GASTRONOMIA,004,1.792373e+15,9.921553e+14,NaN,0.0,9.921553e+14,NaN,0.0


In [4]:
condado_maestro.head(4)

,num,nombre_establecimiento,RUC,CÓDIGO ESTABLECIMIENTO,ESTADO,CÓDIGO ESTABLECIMIENTO.1,id_establecimiento,categoria,tipo_local,posible
0,1,ACCESORIOS CELULARES,1792454700001,6,ABIERTO,006,1792454700001006,TECNOLOGIA,LOCAL REGULAR,NaN
1,2,ACIUM,1793194235001,2,ABIERTO,002,1793194235001002,JOYERÍA,LOCAL REGULAR,NaN
2,3,ADIDAS,1790391795001,NO COINCIDE RUC,ABIERTO,NO COINCIDE RUC,NaN,MODA,LOCAL REGULAR,NaN
3,4,ADRISSA,1792502527001,4,ABIERTO,004,1792502527001004,MODA,LOCAL REGULAR,NaN


In [5]:
malls.head(4)

,numero_ruc,razon_social,nombre_tienda,ciudad,sector,codigo_establecimiento,centro_comercial
0,1.792141e+12,PROMOTORA ECUATORIANA DE CAFE DE COLOMBIA S.A....,JUAN VALDEZ,QUITO,IÑAQUITO,2.0,MALL EL JARDIN
1,1.792141e+12,PROMOTORA ECUATORIANA DE CAFE DE COLOMBIA S.A....,NaN,QUITO,IÑAQUITO,7.0,QUICENTRO
2,1.792141e+12,PROMOTORA ECUATORIANA DE CAFE DE COLOMBIA S.A....,NaN,QUITO,IÑAQUITO,12.0,QUICENTRO
3,1.792141e+12,PROMOTORA ECUATORIANA DE CAFE DE COLOMBIA S.A....,NaN,RUMIÑAHUI,SANGOLQUI,13.0,SAN LUIS


In [6]:
scala_rucs=scala_maestro['ruc'].unique()
condado_rucs=condado_maestro['RUC'].unique()

malls_rucs=malls['numero_ruc'].unique()

In [7]:
rucs = np.concatenate((scala_rucs, condado_rucs,malls_rucs))

In [8]:
rucs=np.unique(rucs)

In [9]:
len(rucs)

279

In [10]:
rucs=rucs[~np.isnan(rucs)]

## Info rucs sri

In [19]:
#Parámetros conexión
conexion = mysql.connector.connect(
    host="dl-radar.cluster-ro-c7pmwdslewrp.us-east-1.rds.amazonaws.com",
    user="debian",
    password="eeAZU3v1FXCY9zmbvcS6kpEpyj",
    database="data_fact",
    port="4408"
)

In [20]:
# Consulta establecimientos
ruc_v="1792342783001"

tabla="base_rucs_sri"
info_sucursales_jv=[]

text=tabla
consulta = "SELECT * FROM " + text + " WHERE estado_establecimiento='ABIERTO' AND numero_ruc IN (" + ",".join([f"'{r}'" for r in rucs]) + ")"
print(consulta)
cursor = conexion.cursor()
cursor.execute(consulta)
registros = cursor.fetchall()
cursor.close()
df = pd.DataFrame(registros, columns=cursor.column_names)
info_sucursales_jv.append(df)
  
info_sucursales_jv = pd.concat(info_sucursales_jv)

SELECT * FROM base_rucs_sri WHERE estado_establecimiento='ABIERTO' AND numero_ruc IN ('102311594001.0','190007510001.0','190055965001.0','190072002001.0','190111881001.0','190115798001.0','190341526001.0','190399524001.0','190413233001.0','190459136001.0','190471810001.0','400865911001.0','590031984001.0','704639889001.0','905374310001.0','916044563001.0','990000530001.0','990005737001.0','990011214001.0','990021058001.0','990043027001.0','990048673001.0','990049459001.0','990379017001.0','990858322001.0','990967946001.0','990987874001.0','991274545001.0','991286403001.0','991408843001.0','991458050001.0','992106891001.0','992141913001.0','992146109001.0','992155272001.0','992262540001.0','992270020001.0','992307676001.0','992358173001.0','992413077001.0','992415290001.0','992472340001.0','992531983001.0','992621702001.0','992622393001.0','992653884001.0','992676868001.0','992704020001.0','992858753001.0','992952423001.0','993041408001.0','993119210001.0','993129496001.0','993205893001

In [21]:
info_sucursales_jv

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,tipo_establecimiento,direccion_completa,fecha_inicio_actividades_comercio,fecha_cese_comercio,fecha_reinicio_actividades_comercio,fecha_actualizacion_comercio,nombre_representante_legal,identificacion_representante_legal,fecha_actualizacion,encontrado
0,102311594001001,102311594001,1,GUEVARA BUESTAN KLEVER HOMERO,JOMATIK,1,ACTIVO,1,ABIERTO,0,...,OFI,AZUAY / CUENCA / SUCRE / REMIGIO TAMARIZ CRESP...,2000-08-11,None,None,2025-09-17,,,2025-10-16 14:29:12,1
1,102311594001004,102311594001,4,GUEVARA BUESTAN KLEVER HOMERO,FOSSIL,1,ACTIVO,1,ABIERTO,0,...,OFI,AZUAY / CUENCA / YANUNCAY / FELIPE SEGUNDO S/N...,2000-08-11,None,None,2025-09-17,,,2025-10-16 14:29:12,1
2,102311594001005,102311594001,5,GUEVARA BUESTAN KLEVER HOMERO,BOSI,1,ACTIVO,1,ABIERTO,1,...,MAT,AZUAY / CUENCA / YANUNCAY / FELIPE SEGUNDO S/N...,2000-08-11,None,None,2025-09-17,,,2025-10-16 14:29:12,1
3,102311594001010,102311594001,10,GUEVARA BUESTAN KLEVER HOMERO,WATCH PLUS MALL DEL SOL,1,ACTIVO,1,ABIERTO,0,...,OFI,GUAYAS / GUAYAQUIL / XIMENA / JUAN TANCA MAREN...,2000-08-11,None,None,2025-09-17,,,2025-10-16 14:29:12,1
4,102311594001011,102311594001,11,GUEVARA BUESTAN KLEVER HOMERO,FOSSIL MALL DEL SOL,1,ACTIVO,1,ABIERTO,0,...,OFI,GUAYAS / GUAYAQUIL / TARQUI / AV. JOAQUIN ORRA...,2000-08-11,None,None,2025-09-17,,,2025-10-16 14:29:12,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9603,1804381307001006,1804381307001,6,RODRIGUEZ GUAÑO SANDRA PAULINA,CELLULAR BOX,1,ACTIVO,1,ABIERTO,0,...,OFI,GUAYAS / GUAYAQUIL / TARQUI / AV. FCO DE ORELL...,2017-06-23,None,None,2025-08-14,,,2025-10-10 21:24:39,1
9604,1804381307001008,1804381307001,8,RODRIGUEZ GUAÑO SANDRA PAULINA,MIKAS MARKET,1,ACTIVO,1,ABIERTO,0,...,OFI,PICHINCHA / QUITO / IÑAQUITO / AV RIO AMAZONAS...,2017-06-23,None,None,2025-08-14,,,2025-10-10 21:24:39,1
9605,1804381307001009,1804381307001,9,RODRIGUEZ GUAÑO SANDRA PAULINA,CELLULAR BOX MATRIZ,1,ACTIVO,1,ABIERTO,0,...,OFI,PICHINCHA / QUITO / IÑAQUITO / AV AMAZONAS L 3...,2017-06-23,None,None,2025-08-14,,,2025-10-10 21:24:39,1
9606,1900026475001004,1900026475001,4,GOMEZ TUZA CARLOTA ANTONIA,FUN RIDES,1,ACTIVO,1,ABIERTO,1,...,MAT,PICHINCHA / QUITO / CHILLOGALLO / AV. QUITUMBE...,2000-10-31,None,2012-07-05,2024-02-28,,,2025-10-10 03:20:24,1


In [22]:
info_sucursales_jv["Provincia"]=info_sucursales_jv["direccion_completa"].str.split(" / ", n=3, expand=True)[0]
info_sucursales_jv["Ciudad"]=info_sucursales_jv["direccion_completa"].str.split(" / ", n=3, expand=True)[1]
info_sucursales_jv["Sector"]=info_sucursales_jv["direccion_completa"].str.split(" / ", n=3, expand=True)[2]
info_sucursales_jv["Direccion"]=info_sucursales_jv["direccion_completa"].str.split(" / ", n=3, expand=True)[3]

In [23]:
def _norm_one(x: str) -> str:
    x = "" if x is None else str(x)
    x = x.lower().strip()
    x = unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('ascii')
    x = re.sub(r'\s+', ' ', x)
    return x

def _norm_series(s: pd.Series) -> pd.Series:
    return s.fillna("").map(_norm_one)

def filtrar_ubicacion_same_schema(
    df: pd.DataFrame,
    provincia: str = None,
    ciudad: str = None,
    sector_like: str = None,
    direccion_like: str = None
) -> pd.DataFrame:

    cols = set(df.columns)
    use_prov = 'Provincia' in cols and provincia
    use_ciud = 'Ciudad' in cols and ciudad
    use_sect = 'Sector' in cols and sector_like
    use_dir  = 'Direccion' in cols and direccion_like

    if use_prov:
        prov_series = _norm_series(df['Provincia'])
        provincia_n = _norm_one(provincia)
    if use_ciud:
        ciud_series = _norm_series(df['Ciudad'])
        ciudad_n = _norm_one(ciudad)
    if use_sect:
        sect_series = _norm_series(df['Sector'])
        sector_n = _norm_one(sector_like)
    if use_dir:
        dir_series = _norm_series(df['Direccion'])
        direccion_n = _norm_one(direccion_like)

    mask = pd.Series(True, index=df.index)

    before = len(df)

    if use_prov:
        m = (prov_series == provincia_n)
        mask &= m
    if use_ciud:
        m = (ciud_series == ciudad_n)
        mask &= m
    if use_sect:

        m = sect_series.str.contains(re.escape(sector_n), na=False)
        mask &= m
    if use_dir:
        m = dir_series.str.contains(re.escape(direccion_n), na=False)
        mask &= m

    result = df.loc[mask].copy()


    return result


In [24]:

filtrado = filtrar_ubicacion_same_schema(
    info_sucursales_jv,
    provincia="MANABI",
    ciudad="MANTA",
    sector_like="MANTA",
    direccion_like="MALECON"
)


In [25]:
# Este revisar
filtrado

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,fecha_reinicio_actividades_comercio,fecha_actualizacion_comercio,nombre_representante_legal,identificacion_representante_legal,fecha_actualizacion,encontrado,Provincia,Ciudad,Sector,Direccion
165,190055965001058,190055965001,58,BANCO DEL AUSTRO S. A,BANCO DEL AUSTRO,1,ACTIVO,1,ABIERTO,0,...,None,2025-08-14,TAMARIZ KLINKICHT ROBERTO CLEMENTE,0102115763,2025-10-15 07:11:53,1,MANABI,MANTA,MANTA,AV. MALECON S/N Y CALLE 16
535,990000530001033,990000530001,33,PYCCA S.A.,PYCCA MANTA,1,ACTIVO,1,ABIERTO,0,...,None,2025-09-24,CALVO RIVAS JUAN LUIS,0960611499,2025-10-15 18:55:12,1,MANABI,MANTA,MANTA,AV. MALECON S/N Y AV. 23 Y CALLE 20
786,990011214001038,990011214001,38,ALMACENES DE PRATI S.A.,DEPRATI,1,ACTIVO,1,ABIERTO,0,...,None,2025-10-03,DALY RODRÍGUEZ EDUARDO,124447909,2025-10-15 18:56:39,1,MANABI,MANTA,MANTA,AV. MALECON S/N Y AV. 23 CALLE 20
860,990049459001063,990049459001,63,BANCO GUAYAQUIL S.A.,BANCO GUAYAQUIL S.A. MANTA,1,ACTIVO,1,ABIERTO,0,...,None,2025-09-01,MACKLIFF ELIZALDE JULIO ANTONIO,0909136251,2025-10-15 18:58:28,1,MANABI,MANTA,MANTA,MALECON S/N Y AV. 14
966,990379017001031,990379017001,31,BANCO BOLIVARIANO C.A.,BANCO BOLIVARIANO AGENCIA MANTA,1,ACTIVO,1,ABIERTO,0,...,None,2025-10-03,VALLARINO MARCOS VICENTE JOSE,0908860018,2025-10-15 19:04:03,1,MANABI,MANTA,MANTA,AV. MALECON JAIME CHAVEZ S/N Y CALLE 14 AVA
3074,991408843001022,991408843001,22,ESVELTS S.A.,ONLY NATURAL,1,ACTIVO,1,ABIERTO,0,...,None,2025-09-30,CORNEJO MONTALVAN CARLOS MANUEL,0930349873,2025-10-15 19:35:53,1,MANABI,MANTA,MANTA,AV. 4 DE NOV. (AV. MALECON) S/N Y AV. 23 Y CAL...
3279,992106891001166,992106891001,166,DULCAFE S.A.,SWEET & COFFEE,1,ACTIVO,1,ABIERTO,0,...,None,2025-09-26,GERENLEGAL S.A..,0992886757001,2025-10-15 19:44:18,1,MANABI,MANTA,MANTA,AV. MALECON PB-K08 Y CALLE 23 Y CALLE 20
3280,992106891001167,992106891001,167,DULCAFE S.A.,SWEET & COFFEE,1,ACTIVO,1,ABIERTO,0,...,None,2025-09-26,GERENLEGAL S.A..,0992886757001,2025-10-15 19:44:18,1,MANABI,MANTA,MANTA,AV. MALECON P2-032-033 Y CALLE 23 Y CALLE 20
3332,992155272001012,992155272001,12,NEXCOL S.A.,WATCH WORLD,1,ACTIVO,1,ABIERTO,0,...,None,2025-10-06,BONNARD BASANTES KARINA MARIA,0911942498,2025-10-15 19:50:01,1,MANABI,MANTA,MANTA,AV MALECON SN Y AV 23 CALLE D
3362,992307676001008,992307676001,8,MARCAS LIDERES MARLID S.A.,SKECHERS,1,ACTIVO,1,ABIERTO,0,...,None,2025-03-31,SALTOS RIZZO CATALINA SILVIA,0909497323,2025-10-15 20:04:32,1,MANABI,MANTA,MANTA,AV 4 DE NOVIEMBRE (AV MALECON) Y AV 23 Y CALLE 20


In [26]:
filtrado.to_excel('revision_mallpacifico.xlsx')

In [ ]:
filtrado.columns

In [ ]:
resultado_grouped = (
    filtrado
    .groupby(['numero_ruc', 'razon_social', 'numero_establecimiento', 'nombre_fantasia_comercial'])
    .size()
    .reset_index(name='conteo')
)

In [ ]:
resultado_grouped